In [ ]:
# loading packages


import duckdb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import transformers
import accelerate
from transformers import BertForSequenceClassification
import time
import torch
import numpy as np
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification
import pickle
import os
import s3fs
from sklearn.metrics import classification_report


In [ ]:
# loading the dataset

con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

query = "SELECT * FROM read_parquet('https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet')"
df = con.sql(query).df()

In [ ]:
# load model and encoders

model_path = "model_nace"
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()
tokenizer = BertTokenizer.from_pretrained(model_path)
with open(f"{model_path}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)
    
def tokenize(batch):
    return tokenizer(
        batch['label'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [ ]:
# loading the dataset

con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

query = "SELECT * FROM read_parquet('https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet')"
df = con.sql(query).df()
df['target'] = le.fit_transform(df['code'])

num_labels = df['target'].nunique()
print(num_labels)
df.head()


In [ ]:

# create train and test dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['target'],  # class equilibtrate
    random_state=42
)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:

# conversion in datasets
test_dataset = Dataset.from_pandas(test_df[['label', 'target']])

test_dataset = test_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.rename_column("target", "labels")

test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


# temporary dataset reduced
#test_dataset = test_dataset.shuffle().select(range(4))


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    disable_tqdm=False,   # 🔥 important
    report_to="none"     
)
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
predictions = trainer.predict(test_dataset)

In [ ]:
# accuracy on the test dataset

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))



In [ ]:
# classification report
print(classification_report(y_true, y_pred))
y_pred_code = le.inverse_transform(y_pred)
y_true_code = le.inverse_transform(y_true)

In [ ]:
# compare prediction and true class
df_results = test_df.copy().reset_index(drop=True)

df_results["true_code"] = y_true_code
df_results["pred_code"] = y_pred_code

In [ ]:

nace = pd.read_csv(f"{model_path}/nace.csv", dtype={'CODE': str})
nace.head()

In [ ]:
nace = nace[['CODE', 'HEADING']]
df_results = df_results.merge(
    nace.rename(columns={"CODE": "true_code", "HEADING": "true_name"}),
    on="true_code",
    how="left"
)
df_results = df_results.merge(
    nace.rename(columns={"CODE": "pred_code",  "HEADING": "pred_name"}),
    on="pred_code",
    how="left"
)
df_results


In [ ]:
#errors

df_errors = df_results[df_results["true_code"] != df_results["pred_code"]]
df_errors.head(20)

In [ ]:

def predict_label(text, model, tokenizer, le, nomenclature, device):
    # tokenisation
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # envoyer sur device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # prédiction
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_class_id = torch.argmax(logits, dim=1).cpu().numpy()[0]

    # décodage du label
    pred_code = le.inverse_transform([pred_class_id])[0]

    # récupérer le libellé associé
    pred_name = nace.loc[
        nace["CODE"] == pred_code, "HEADING"
    ].values[0]

    return {
        "text": text,
        "pred_class_id": pred_class_id,
        "pred_code": pred_code,
        "pred_name": pred_name
    }

    
# device (GPU si dispo)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
text = "Business Development Executive"

result = predict_label(text, model, tokenizer, le, nace, device)

print("Text :", result["text"])
print("Classe prediction :", result["pred_class_id"])
print("Code prediction :", result["pred_code"])
print("name prediction :", result["pred_name"])

In [ ]:
df_errors.to_csv("./model_nace/df_errors.csv", index=False)